# LIGHT GraphRAG 데이터 전처리: King Top Persona Episode 추출

이 노트북은 `agents.name == "king"`인 agent들을 `persona`별로 집계한 뒤, **episode 수가 가장 많은 king persona**에 해당하는 episode만 추출하여 GraphRAG 후속 구축용 전처리 산출물을 생성합니다.

이번 전처리의 목표는 전체 king 관련 episode가 아니라, **동일 persona를 가진 king character episode 332개**를 중심으로 일관된 그래프/문서 데이터를 만드는 것입니다.


## 1. 폴더 구조

요청한 기준 폴더 구조는 다음과 같습니다.

```text
project_root/
├─ data/
│  ├─ light_data.pkl
│  ├─ light_unseen_data.pkl
│  └─ light_environment.pkl
└─ data_prep/
   ├─ light_king_top_persona_prep.py
   ├─ light_king_top_persona_prep.ipynb
   └─ OUTPUT/
      ├─ metrics/
      ├─ subsets/
      ├─ neo4j/
      ├─ graphrag/
      └─ preprocessing_summary.json
```

이 노트북은 `data_prep/` 폴더에서 실행하는 것을 기준으로 작성되었습니다.


In [1]:
from pathlib import Path
import json
import pandas as pd

# 전처리 로직은 같은 폴더의 py 파일에서 import합니다.
from light_king_top_persona_prep import (
    DEFAULT_RAW_DIR,
    DEFAULT_OUT_DIR,
    iter_episodes,
    count_agent_personas,
    build_outputs,
    validate_outputs,
)

RAW_DIR = DEFAULT_RAW_DIR
OUT_DIR = DEFAULT_OUT_DIR

print("RAW_DIR:", RAW_DIR)
print("OUT_DIR:", OUT_DIR)
print("RAW files exist:")
for name in ["light_data.pkl", "light_unseen_data.pkl", "light_environment.pkl"]:
    print(" -", name, (RAW_DIR / name).exists())

RAW_DIR: D:\encore\교과목-2 프로젝트\mle-01-p2-team3\data
OUT_DIR: D:\encore\교과목-2 프로젝트\mle-01-p2-team3\data_prep\OUTPUT
RAW files exist:
 - light_data.pkl True
 - light_unseen_data.pkl True
 - light_environment.pkl True


## 2. King agent persona별 episode 수 확인

먼저 전체 train/unseen episode에서 `agents.name == "king"`인 agent를 찾고, `persona`별로 episode 수를 집계합니다.

- `episode_count`: 해당 persona를 가진 king agent가 등장한 episode 수
- `agent_count`: agents 목록에서 해당 persona의 king agent가 등장한 횟수

이번 전처리 대상은 `episode_count`가 가장 큰 persona입니다.


In [2]:
episodes = iter_episodes(RAW_DIR)
print("total episodes:", len(episodes))

king_persona_counts_df = count_agent_personas(episodes, target_agent_name="king")
king_persona_counts_df.head(20)

total episodes: 11024


,persona_id,agent_name,persona,episode_count,agent_count,example_episode_id,example_split,example_setting_name,example_setting_category,rank
0,persona__f9e1d9295997,king,I am a king of the whole empire. I give rules ...,332,332,train_ep_001422,train,Royal Gardens,Outside Palace,1
1,persona__37b5382c20af,king,I am the king. I rule this kingdom alongside m...,291,291,train_ep_000046,train,Black Smiths shop,Town,2
2,persona__c5867370016d,king,I rule over many people. I spend my days in le...,108,108,train_ep_001372,train,The Great Hall,Inside Tower,3
3,persona__44a8766492ce,king,"I am the king of my lands, or, I used to be. I...",46,46,unseen_ep_000015,unseen,The dungeon,underwater aquapolis,4
4,persona__e81ee2a30fdd,king,I am the king of a small province. I am rotund...,38,38,train_ep_001388,train,Master Bedroom,Inside Palace,5
5,persona__4161d99ca83d,king,I am a king who was destined to rule. I am kno...,31,31,train_ep_000136,train,palace ballroom,Inside Palace,6
6,persona__82f87f52df89,king,I was born into power. I was raised to rule an...,31,31,train_ep_000158,train,Master Bedroom,Inside Castle,7
7,persona__9b3c93ae5ab5,king,I am the most powerful man in the country. I l...,20,20,train_ep_000076,train,Across the King's Garden,Trail,8
8,persona__d062ce11e9ee,king,I am a King who rules a vast and mighty land. ...,13,13,train_ep_000547,train,The room at the top of the tower,Inside Castle,9
9,persona__141f7ceb5007,king,I am the king of this great land. I collect ma...,12,12,train_ep_000091,train,Weapon Closet,Inside Tower,10


In [3]:
top_persona_row = king_persona_counts_df.iloc[0]
print("선택될 top persona")
print("persona_id:", top_persona_row["persona_id"])
print("episode_count:", top_persona_row["episode_count"])
print("agent_count:", top_persona_row["agent_count"])
print("persona:", top_persona_row["persona"])

선택될 top persona
persona_id: persona__f9e1d9295997
episode_count: 332
agent_count: 332
persona: I am a king of the whole empire. I give rules and pursuit them. I am brave and fearless.


## 3. 전체 전처리 실행

아래 셀은 top king persona episode만 대상으로 전체 전처리를 실행합니다.

생성되는 산출물은 다음과 같습니다.

```text
OUTPUT/
├─ metrics/
│  ├─ king_agent_persona_episode_counts.csv
│  ├─ target_persona_episode_metrics.csv
│  ├─ top_characters_in_target_persona_episodes.csv
│  ├─ target_persona_locations.csv
│  ├─ target_persona_objects.csv
│  ├─ target_persona_cocharacters.csv
│  └─ target_persona_summary.json
├─ subsets/
│  └─ target_king_persona_episodes.jsonl
├─ neo4j/
│  ├─ nodes_*.csv
│  └─ edges_*.csv
└─ graphrag/
   └─ graphrag_documents_target_king_persona.jsonl
```

`clean=True`는 기존 `OUTPUT` 폴더를 삭제하고 새로 생성합니다.


In [4]:
summary = build_outputs(
    raw_dir=RAW_DIR,
    out_dir=OUT_DIR,
    target_agent_name="king",
    target_persona=None,  # None이면 episode_count 1위 persona를 자동 선택합니다.
    clean=True,
)

summary

{'characters': 264,
 'personas': 1,
 'episodes': 332,
 'utterances': 4216,
 'locations': 39,
 'objects': 96,
 'edges_appears_in': 662,
 'edges_has_target_persona': 332,
 'edges_speaks': 4209,
 'edges_mentions': 2600,
 'edges_occurs_in': 332,
 'edges_contains_object': 487,
 'edges_co_occurs_with': 70,
 'target_persona_episode_jsonl': 332,
 'graphrag_documents_target_persona': 332,
 'target_agent_name': 'king',
 'target_persona_id': 'persona__f9e1d9295997',
 'target_persona': 'I am a king of the whole empire. I give rules and pursuit them. I am brave and fearless.',
 'target_episode_count': 332,
 'raw_total_episode_count': 11024}

## 4. 최종 검증

전처리 후 다음을 검증합니다.

1. node ID 중복 여부
2. edge가 참조하는 node ID가 실제 node CSV에 존재하는지 여부
3. target persona episode JSONL 수와 GraphRAG document 수 일치 여부


In [5]:
validated_summary = validate_outputs(OUT_DIR)
validated_summary

{'characters': 264,
 'personas': 1,
 'episodes': 332,
 'utterances': 4216,
 'locations': 39,
 'objects': 96,
 'edges_appears_in': 662,
 'edges_has_target_persona': 332,
 'edges_speaks': 4209,
 'edges_mentions': 2600,
 'edges_occurs_in': 332,
 'edges_contains_object': 487,
 'edges_co_occurs_with': 70,
 'target_persona_episode_jsonl': 332,
 'graphrag_documents_target_persona': 332}

## 5. 주요 산출물 확인

전처리 결과 중 핵심 파일을 확인합니다.


In [6]:
metrics_dir = OUT_DIR / "metrics"
neo4j_dir = OUT_DIR / "neo4j"
graphrag_dir = OUT_DIR / "graphrag"

summary_path = OUT_DIR / "preprocessing_summary.json"
with summary_path.open("r", encoding="utf-8") as f:
    preprocessing_summary = json.load(f)

preprocessing_summary

{'characters': 264,
 'personas': 1,
 'episodes': 332,
 'utterances': 4216,
 'locations': 39,
 'objects': 96,
 'edges_appears_in': 662,
 'edges_has_target_persona': 332,
 'edges_speaks': 4209,
 'edges_mentions': 2600,
 'edges_occurs_in': 332,
 'edges_contains_object': 487,
 'edges_co_occurs_with': 70,
 'target_persona_episode_jsonl': 332,
 'graphrag_documents_target_persona': 332,
 'target_agent_name': 'king',
 'target_persona_id': 'persona__f9e1d9295997',
 'target_persona': 'I am a king of the whole empire. I give rules and pursuit them. I am brave and fearless.',
 'target_episode_count': 332,
 'raw_total_episode_count': 11024}

In [7]:
target_metrics_df = pd.read_csv(metrics_dir / "target_persona_episode_metrics.csv")
print("target episode count:", len(target_metrics_df))
target_metrics_df.head()

target episode count: 332


,episode_id,split,target_agent_name,target_persona_id,target_persona,target_persona_agent_count_in_episode,utterance_count,agent_count
0,train_ep_001422,train,king,persona__f9e1d9295997,I am a king of the whole empire. I give rules ...,1,14,2
1,train_ep_001438,train,king,persona__f9e1d9295997,I am a king of the whole empire. I give rules ...,1,14,2
2,train_ep_001493,train,king,persona__f9e1d9295997,I am a king of the whole empire. I give rules ...,1,14,2
3,train_ep_001494,train,king,persona__f9e1d9295997,I am a king of the whole empire. I give rules ...,1,14,2
4,train_ep_001501,train,king,persona__f9e1d9295997,I am a king of the whole empire. I give rules ...,1,14,2


In [8]:
characters_df = pd.read_csv(metrics_dir / "top_characters_in_target_persona_episodes.csv")
characters_df.head(20)

,character_id,normalized_name,canonical_name,appearance_count,speaker_turn_count,mention_count,unique_episode_count,total_score,is_target_agent_name,is_generic_name,source,overall_rank_in_target_subset
0,char__king,king,King,332,2117,447,332,3560,True,False,top_king_persona_subset,1
1,char__queen,queen,Queen,44,281,143,101,556,False,False,top_king_persona_subset,2
2,char__servant,servant,Servant,33,188,53,45,340,False,True,top_king_persona_subset,3
3,char__maid,maid,Maid,17,112,30,24,193,False,False,top_king_persona_subset,4
4,char__guard,guard,Guard,15,96,52,36,193,False,True,top_king_persona_subset,5
5,char__knight,knight,Knight,13,90,33,20,162,False,False,top_king_persona_subset,6
6,char__people,people,People,3,16,109,65,134,False,True,top_king_persona_subset,7
7,char__brother,brother,Brother,10,70,33,21,133,False,False,top_king_persona_subset,8
8,char__townperson,townperson,Townperson,14,88,2,14,132,False,False,top_king_persona_subset,9
9,char__guest,guest,Guest,12,80,5,12,121,False,True,top_king_persona_subset,10


In [9]:
nodes_episodes_df = pd.read_csv(neo4j_dir / "nodes_episodes.csv")
nodes_characters_df = pd.read_csv(neo4j_dir / "nodes_characters.csv")
nodes_personas_df = pd.read_csv(neo4j_dir / "nodes_personas.csv")

print("episodes:", len(nodes_episodes_df))
print("characters:", len(nodes_characters_df))
print("personas:", len(nodes_personas_df))
nodes_personas_df

episodes: 332
characters: 264
personas: 1


,persona_id,agent_name,persona,episode_count,is_selected_target_persona
0,persona__f9e1d9295997,king,I am a king of the whole empire. I give rules ...,332,True


## 6. GraphRAG 문서 샘플 확인

`graphrag_documents_target_king_persona.jsonl`은 후속 팀원이 청킹/임베딩을 수행할 입력 문서입니다.


In [10]:
doc_path = graphrag_dir / "graphrag_documents_target_king_persona.jsonl"

with doc_path.open("r", encoding="utf-8") as f:
    first_doc = json.loads(next(f))

print("doc_id:", first_doc["doc_id"])
print("episode_id:", first_doc["episode_id"])
print("title:", first_doc["title"])
print("metadata keys:", list(first_doc["metadata"].keys()))
print("\n--- text preview ---")
print(first_doc["text"][:2000])

doc_id: doc__train_ep_001422
episode_id: train_ep_001422
title: Top king persona episode in Royal Gardens
metadata keys: ['split', 'setting_name', 'setting_normalized_name', 'setting_category', 'agents', 'agent_personas', 'top_objects', 'utterance_count', 'target_agent_name', 'target_persona_id', 'target_persona']

--- text preview ---
Episode ID: train_ep_001422
Split: train
Target Agent: king
Target Persona: I am a king of the whole empire. I give rules and pursuit them. I am brave and fearless.

[Setting]
Name: Royal Gardens
Category: Outside Palace
Description: Lined with rose bushes that look as if they have been watered by the God's, the Royal Gardens is a beauty to behold. An intricate labyrinth made of shrubs is at the center ending with a fountain. There are various benches on the sides of the rose bushes and a small lake in the back drop.
Background: The Royal Gardens were conceptualized by the late great Queen. She wanted a beautiful promenade to walk her dogs and converse w

## 7. CLI 실행 참고

노트북 대신 `.py` 파일을 직접 실행할 경우 `data_prep/` 폴더에서 다음 명령을 사용합니다.

```bash
python light_king_top_persona_prep.py --clean
```

특정 persona를 직접 지정하고 싶다면 다음처럼 실행합니다.

```bash
python light_king_top_persona_prep.py --target-persona "I am a king of the whole empire. I give rules and pursuit them. I am brave and fearless." --clean
```
